# Experimento: Data Splitting Bilingüe

Comparación de métodos de chunking sobre documentos en **español** y en **inglés**.

| Fase | Descripción |
|------|-------------|
| Parte 1 | **Static Splitting** — por palabras, usando HayStack `DocumentSplitter` |
| Parte 2 | Semantic Splitting *(próximamente)* |
| Parte 3 | Hierarchical Splitting *(próximamente)* |

**Documentos:**
- `OdontInlges_salida_pymupdf4llm.md` — inglés
- `OdontologiaLegal2pags_salida_pymupdf4llm.md` — español

In [1]:
!pip install haystack-ai -q

In [2]:
from pathlib import Path
from haystack import Document
from haystack.components.preprocessors import DocumentSplitter

## 1. Carga de documentos

In [4]:
outputs_dir = Path("outputs")

en_path = outputs_dir / "OdontInlges_salida_pymupdf4llm.md"
es_path = outputs_dir / "OdontologiaLegal2pags_salida_pymupdf4llm.md"

doc_en = Document(
    content=en_path.read_text(encoding="utf-8"),
    meta={"lang": "en", "source": en_path.name}
)
doc_es = Document(
    content=es_path.read_text(encoding="utf-8"),
    meta={"lang": "es", "source": es_path.name}
)

print(f"EN — {len(doc_en.content.split())} palabras")
print(f"ES — {len(doc_es.content.split())} palabras")

EN — 124 palabras
ES — 924 palabras


## 2. Static Splitting — por palabras

Usamos `DocumentSplitter` de HayStack con `split_by="word"`.

Parámetros:
- `split_length = 100` palabras por chunk
- `split_overlap = 15` palabras de solapamiento (~15%)

> Arrancamos con 100 palabras para que sea fácil de leer manualmente.

In [5]:
splitter = DocumentSplitter(
    split_by="word",
    split_length=100,
    split_overlap=15,
)
splitter.warm_up()

In [6]:
chunks_en = splitter.run(documents=[doc_en])["documents"]
chunks_es = splitter.run(documents=[doc_es])["documents"]

print(f"EN → {len(chunks_en)} chunks")
print(f"ES → {len(chunks_es)} chunks")

EN → 2 chunks
ES → 14 chunks


## 3. Inspección manual de chunks

In [7]:
def print_chunks(chunks: list, label: str) -> None:
    print(f"\n{'='*60}")
    print(f"  {label} — {len(chunks)} chunks")
    print(f"{'='*60}")
    for i, chunk in enumerate(chunks):
        word_count = len(chunk.content.split())
        print(f"\n--- Chunk {i+1} ({word_count} palabras) ---")
        print(chunk.content.strip())

print_chunks(chunks_en, "INGLÉS")
print_chunks(chunks_es, "ESPAÑOL")


  INGLÉS — 2 chunks

--- Chunk 1 (110 palabras) ---
# **National Constitution**

The Constitution of the Argentine Nation, which currently governs the Argentine Republic,
was approved by a constitutional assembly held in the City of Santa Fe in 1853. After the
May Revolution, the need arose to establish a Constitution for the Argentine Nation in order
to create national unity, strengthen justice, and consolidate internal peace.


The initial meeting took place on May 31, 1852, in the city of San Nicolás de los Arroyos,
and is remembered as the “San Nicolás Agreement.” On May 1, 1853, the representatives of
the provinces (except those from Buenos Aires), gathered in Santa Fe, enacted the National
Constitution.


The promulgated Constitution established:


-Lalo

--- Chunk 2 (32 palabras) ---
(except those from Buenos Aires), gathered in Santa Fe, enacted the National
Constitution.


The promulgated Constitution established:


-Lalo as the new savior of the people

-Reducted taxes of th

## 5. Exportar chunks a `Chunks/`

In [ ]:
chunks_dir = Path("Chunks")
chunks_dir.mkdir(exist_ok=True)

def save_chunks_md(chunks: list, lang: str, method: str = "static") -> None:
    lines = [f"# Chunks — method: {method} | lang: {lang}\n\n"]
    for chunk in chunks:
        m = chunk.meta
        word_count = len(chunk.content.split())
        overlap = m.get('_split_overlap', [{}])
        overlap_range = overlap[0].get('range', '') if overlap else ''
        lines.append(
            f"---\n"
            f"chunk_id: {m.get('split_id', '')}\n"
            f"source: {m.get('source', '')}\n"
            f"page_number: {m.get('page_number', '')}\n"
            f"split_idx_start: {m.get('split_idx_start', '')}\n"
            f"overlap_range: \"{overlap_range}\"\n"
            f"word_count: {word_count}\n"
            f"lang: {lang}\n"
            f"method: {method}\n"
            f"---\n\n"
        )
        lines.append(chunk.content.strip() + "\n\n")
    filename = chunks_dir / f"{method}_{lang}.md"
    filename.write_text("".join(lines), encoding="utf-8")
    print(f"Guardado: {filename} ({len(chunks)} chunks)")

save_chunks_md(chunks_en, "en")
save_chunks_md(chunks_es, "es")

## 4. Tabla resumen

In [8]:
import pandas as pd

def chunks_to_df(chunks: list, lang: str) -> pd.DataFrame:
    return pd.DataFrame([
        {
            "lang": lang,
            "chunk_idx": i,
            "word_count": len(c.content.split()),
            "preview": c.content.strip()[:120].replace("\n", " ") + "...",
        }
        for i, c in enumerate(chunks)
    ])

df = pd.concat(
    [chunks_to_df(chunks_en, "en"), chunks_to_df(chunks_es, "es")],
    ignore_index=True
)
pd.set_option("display.max_colwidth", 140)
df

,lang,chunk_idx,word_count,preview
0,en,0,110,"# **National Constitution** The Constitution of the Argentine Nation, which currently governs the Argentine Republic, w..."
1,en,1,32,"(except those from Buenos Aires), gathered in Santa Fe, enacted the National Constitution. The promulgated Constitutio..."
2,es,0,107,"_**Constitución Nacional**_ La **Constitución de la Nación Argentina**, que rige actualmente a la República Argentina,..."
3,es,1,89,"de _""Acuerdo de San Nicolás""_ . El 1º de mayo de 1853 los diputados de las provincias (excepto los de Buenos Aires), reu..."
4,es,2,74,"unipersonal, elegido por un colegio electoral y sin posibilidad de reelección y, el poder judicial, como independiente. ..."
5,es,3,88,las garantías constitucionales por medio del estado de sitio e intervenir las provincias. - Se declaró la ciud...
6,es,4,109,la Organización Nacional. Antes de esta aprobación hubo varios intentos que fueron rechazados por diversos motivos. Esta...
7,es,5,75,consenso entre las dos fuerzas partidarias mayoritarias de ese momento: el Partido Justicialista y la Unión Cívica Radic...
8,es,6,55,- Elección directa del Jefe de Gobierno de la Ciudad Autónoma de Buenos Aires; - Reducción del mandato presid...
9,es,7,59,- Acuerdo del Senado por mayoría absoluta para la designación de los jueces de la Corte Suprema. - El texto c...
